<a href="https://colab.research.google.com/github/shoukk8-afk/Architecture-of-Diffusion-by-PyTorch/blob/main/Diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms, datasets
import math

In [27]:
#線形拡散スケジューリング
def linear_diffusion_schedule(diffusion_times):
    diffusion_times = torch.tensor(diffusion_times)
    min_rate = 0.0001
    max_rate = 0.02
    betas = min_rate + torch.tensor(diffusion_times) * (max_rate - min_rate)
    alphas = 1 - betas
    alpha_bars = torch.cumprod(alphas)
    signal_rates = alpha_bars
    noise_rates = 1 - alpha_bars
    noise_rates = noise_rates.view(1, 1, 1, 1)
    signal_rates = signal_rates.view(1, 1, 1, 1)
    return noise_rates, signal_rates

In [28]:
#コサイン拡散スケジューリング
def cosine_diffusion_schedule(diffusion_times):
    diffusion_times = torch.tensor(diffusion_times)
    signal_rates = torch.cos(diffusion_times * math.pi / 2)
    noise_rates = torch.sin(diffusion_times * math.pi / 2)
    noise_rates = noise_rates.view(1, 1, 1, 1)
    signal_rates = signal_rates.view(1, 1, 1, 1)
    return noise_rates, signal_rates

#オフセット付きコサイン拡散スケジューリング
def offset_cosine_diffusion_schedule(diffusion_times):
    diffusion_times = torch.tensor(diffusion_times)
    min_signal_rate = 0.02
    max_signal_rate = 0.05
    start_angle = torch.acos(max_signal_rate)
    end_angle = torch.acos(min_signal_rate)

    diffusion_angles = start_angle + diffusion_times * (end_angle - start_angle)

    signal_rates = torch.cos(diffusion_angles)
    noise_rates = torch.sin(diffusion_angles)
    noise_rates = noise_rates.view(1, 1, 1, 1)
    signal_rates = signal_rates.view(1, 1, 1, 1)
    return noise_rates, signal_rates

In [23]:
#正弦波埋め込み
def sinusoidal_embedding(t, embedding_dim=32):
    half_dim = embedding_dim // 2

    # 指数スケールで周波数を作成
    emb = math.log(10000) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
    emb = t[:, None].float() * emb[None, :]

    # sinとcosを並べて [batch_size, embedding_dim] に
    embeddings = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
    return embeddings.view(-1, embedding_dim, 1, 1)

In [5]:
#残差ブロック
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, in_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(in_channels, out_channels, 3, padding=1)

    def forward(self, x):
        x = self.bn(x)
        identity = F.relu(self.conv2(x))
        out = F.relu(self.conv2(F.relu(self.conv1(x))))
        out = F.relu(out + identity)
        return out

In [6]:
#ダウンブロック
class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.residual = ResidualBlock(in_channels, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=2)

    def forward(self, x):
        out1 = self.residual(x)
        out2 = self.residual(self.residual(x))
        out3 = F.relu(self.conv(self.residual(self.residual(x))))

        return [out1, out2, out3]

In [7]:
#アップブロック
class Upblock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.residual1 = ResidualBlock(in_channels, out_channels)
        self.residual2 = ResidualBlock(2 * out_channels, out_channels)

    def forward(self, x, x_list):
        x = F.interpolate(x, scale_factor=2, mode='nearest', align_corners=False)
        x = torch.cat((x, x_list[1]), dim=1)
        x = self.residual1(x)
        x = torch.cat((x, x_list[0]), dim=1)
        x = self.residual2(x)
        return x

In [24]:
#U-Net
class UNet(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 32, 1, padding=1)
        self.downblock1 = DownBlock(32, 64)
        self.downblock2 = DownBlock(64, 96)
        self.downblock3 = DownBlock(96, 128)
        self.residual = ResidualBlock(128, 128)
        self.upblock1 = Upblock(224, 96)
        self.upblock2 = Upblock(160, 64)
        self.upblock3 = Upblock(96, 32)
        self.conv3 = nn.Conv2d(32, 3, 3, padding=1)

    def forward(self, x, t):
        step_embedding = sinusoidal_embedding(t)
        step_embedding = F.interpolate(step_embedding, scale_factor=64, mode='nearest', align_corners=False)
        x = F.relu(self.conv1(x))
        out = torch.concat((x, step_embedding))
        out = F.relu(self.conv2(out))
        out1 = self.downblock1(out)
        out2 = self.downblock2(out)
        out3 = self.downblock3(out)
        out = self.residual(self.residual(out))
        out = self.upblock1(out, out3)
        out = self.upblock2(out, out2)
        out = self.upblock3(out, out1)
        out = self.conv3(out)

        return out

In [25]:
#順方向の拡散過程の計算
def get_xt(x, t, noise):
    noise_rates, signal_rates = offset_cosine_diffusion_schedule(t)
    #noise_ratesとsignal_ratesは1次元のテンソルのため、形状を変形
    noise_rates = noise_rates.view(1, 1, 1, 1)
    signal_rates = signal_rates.view(1, 1, 1, 1)
    x_diffuse = signal_rates * x + noise_rates * noise
    return x_diffuse

In [13]:
model = UNet(3)

#GPUに設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

UNet(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
  (downblock1): DownBlock(
    (residual): ResidualBlock(
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2))
  )
  (downblock2): DownBlock(
    (residual): ResidualBlock(
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (conv1): Conv2d(64, 96, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2))
  )
  (downblock3): DownBlock(
    (residua

In [35]:
#訓練ループの構築
def training_loop(n_epochs, optimizer, model, loss_fn, train_loader, test_loader,
                  diffusion_times):
    for epoch in range(1, n_epochs + 1):
        total = 0
        loss_train = 0
        model.train()
        for imgs, _ in train_loader:
            #ノイズとステップのサンプリング
            noise = torch.randn_like(imgs)
            t = torch.randint(0, diffusion_times, (imgs.size(0), ))
            #imgs, t, noiseをGPUに移す
            imgs = imgs.to(device=device)
            t = t.to(device=device)
            noise = noise.to(device=device)

            imgs_diffuse = get_xt(imgs, t, noise)
            noise_predict = model(imgs_diffuse, t)
            loss = loss_fn(noise_predict, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            #1回のループの損失を加算して訓練データの1エボックの損失の合計を出す
            loss_train += loss.item()

        print(f"Epoch: {epoch}, Loss: {loss_train / len(train_loader)}")


In [31]:
#tステップ目からt-1ステップ目の画像の推定
def denoise(img, noise_predict, step):
    noise_rates, signal_rates = offset_cosine_diffusion_schedule(step)
    noise_prestep, signal_prestep = offset_cosine_diffusion_schedule(step-1)
    img_nonoise = (img - noise_rates * noise_predict) / signal_rates
    #t-1ステップからtステップ目のノイズの割合を計算する
    beta_step = 1 - torch.square(signal_rates / signal_prestep)
    #平均の計算
    mu = signal_prestep * beta_step * img_nonoise / torch.square(noise_rates) + signal_rates * torch.square(noise_prestep) * img / torch.square(noise_rates)
    if step >= 2:
        #分散の計算
        std = torch.sqrt((noise_prestep / noise_rates) * beta_step)
        #εのサンプリング
        epsilon = torch.randn_like(img)
        #t-1ステップ目の画像の計算
        img_denoise = mu + std * epsilon
    else:
        img_denoise = mu
    return img_denoise

In [34]:
#逆方向の拡散過程
def inverse(steps, model, length):
    img_noise = torch.randn(1, 3, length, length)
    for step in range(steps, 1, -1):
        noise_predict = model(img_noise, torch.tensor([step], device=device))
        img_noise = denoise(img_noise, noise_predict, step)
    img = img_noise
    return img